This notebook is designed to load neutron data and any non-neutron data (i.e. pressure, current, etc.) in an experimental folder, time bin it, and export it to CSV.

## Initialization

In [ ]:
# Importing needed code

import re
import json
from collections import defaultdict
from functools import reduce
from typing import (
    Callable,
    # TypeVar,
    # Any,
    Literal
)
from datetime import datetime, timezone, timedelta
from math import sqrt, log, pi
import shutil
from pathlib import Path
from enum import Enum

import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
import pandas as pd
import numpy as np
from pint import Quantity
from scipy.signal import find_peaks, peak_widths, peak_prominences

from data_processing.paths import (
    get_report_root, get_exp_root, get_reactor_data_root)
from data_processing.dataframe_validation import (
    DetectorDataframeColumn,
    BinningDataframeColumn,
    NonReactorDataframeColumn,
    SliceFitDataframeColumn
)
from data_processing.experiment_data_keys import (
    ExperimentDataKey,
    ExperimentNeutronData
)
from data_processing.loading.dataframe_loading import load_parquet_psd
from data_processing.loading.timetag_processing import (
    calculate_timetag_hours,
    calculate_event_time
)
# from data_processing.processing.bimodal_fitting import (
#     get_psd_energy_histogram,
#     scan_histogram_slices,
#     find_failed_slices,
#     BimodalBounds,
#     BimodalParams
# )
from data_processing.processing.slice_fitting import (
    get_psd_energy_histogram, scan_histogram_slices, find_failed_slices)
from data_processing.processing.calibration import Detector, recalibrate
from data_processing.processing.neutron_classification import classify
from data_processing.reporting.plotting import plot_scatter, plot_classification
from data_processing import helpers
from data_processing.processing.neutron_window_strategy.strategy_factory import \
    NeutronStrategyFactory
from data_processing.processing.neutron_window_strategy.abstract_strategy import \
    AbstractNeutronStrategy
from data_processing.types import (
    NasaGenerationSettings,
    NeutronDistributionGenerationSettings,
    NeutronWindowSettings,
    WindowType,
    SliceFitStyle,
    BimodalBounds,
    BimodalParams
)
from data_processing.loading.window_loading import (
    load_side_borders, get_neutron_window_paths)
from data_processing.loading.spectrum_unfolding import load_neutron_response_matrix
from data_processing.helpers.get_midpoints_from_bins import get_midpoints_from_bins
from data_processing.processing.spectrum_unfolding import NDHistogram, unfold_spectrum, weight_factor, stopping_criteria, _nan_divide, cut_low_l, UnfoldingProcessInfo

In [ ]:
bins_min = helpers.get_input_with_default(
    "Enter minimum light output (in MeVee), or press Enter for default (0 MeVee)",
    0,
    float
)
bins_max = helpers.get_input_with_default(
    "Enter maximum light output (in MeVee), or press Enter for default (1.2 MeVee)",
    1.2,
    float
)
bins_width = helpers.get_input_with_default(
    "Enter light output bin width (in MeVee), or press Enter for default (0.01 MeVee)",
    0.01,
    float
)

L_bins = np.arange(bins_min, bins_max + bins_width, bins_width).tolist()

In [ ]:
R = load_neutron_response_matrix(
    # Path("response_matrix_hi_res"),
    # Path("response_matrix_4k"),
    # Path("response_matrix_50keV_sigma"),
    # Path("response_matrix_50keV_sigma_NoFWHM"),
    # Path("response_matrix_R3"),
    # Path("response_matrix_R4_sigma_50keV"),
    # Path("response_matrix_R4_sigma_50keV_FWHM"),
    Path("response_matrix_R4_mono"),
    min_L=bins_min,
    max_L=bins_max,
    L_bin_widths=bins_width
)

In [ ]:
R.counts.shape

In [ ]:
np.sum(R.counts, axis=0)

In [ ]:
# class SimType(Enum):
#     TWOFOURFIVE = 1
#     AMBE = 2
#     DDFUSION = 3

# sim_type = helpers.get_input_required(
#     """\
# Which test data should be used?
# 1: 2.45 MeV neutron sim
# 2: AmBe sim
# 3: D-D fusion sim
# """,
#     [SimType.TWOFOURFIVE, SimType.AMBE, SimType.DDFUSION],
#     lambda x: SimType(int(x))
# )

In [ ]:
base_path = Path("unfolding_test")
# test_file = base_path / "output_2.45 MeV.txt"
# test_file = base_path / "sim_2.450_MeV_Tbird_NoRes_Bin1000.csv.npy"
# test_file = base_path / "mono.csv.npy"
test_file = base_path / "mono_FWHM.csv.npy"
# test_file = base_path / "E1.csv.npy"
# test_file = base_path / "E3.csv.npy"

L_array = np.load(test_file)

bins = np.arange(bins_min, bins_max + bins_width, bins_width)
# cut = pd.cut(df["det_pulse (MeVee)"], bins.tolist())
# cut_index = cut.cat.categories
# new_df = pd.DataFrame(
#     df["NPS"].groupby(cut, observed=True).sum().reindex(cut_index, fill_value=0)
# )
np_cps, *_ = np.histogram(L_array, bins=bins)

# old_index = new_df.index
# if not isinstance(old_index, pd.IntervalIndex):
#     raise RuntimeError(
#         f"DataFrame created from cut/groupby for {test_file.name} "
#         + "was not an IntervalIndex as expected"
#     )
# mids = old_index.mid.to_series(index=cut_index)

# np_cps = new_df["NPS"].to_numpy().reshape(-1, 1)
# np_Ls = mids.to_numpy()
np_cps = np_cps.reshape(-1, 1)
np_Ls = (bins[1:] + bins[:-1]) / 2

monoN = NDHistogram(np_cps, [np_Ls, np.ones(1)])

In [ ]:
base_path = Path("unfolding_test")
# test_file = base_path / "output_dd.txt"
# test_file = base_path / "sigma_50keV.csv.npy"
test_file = base_path / "sigma_50keV_FWHM.csv.npy"
# test_file = base_path / "E2.csv.npy"
# test_file = base_path / "E4.csv.npy"

L_array = np.load(test_file)

bins = np.arange(bins_min, bins_max + bins_width, bins_width)

# cut = pd.cut(df["det_pulse (MeVee)"], bins.tolist())
# cut_index = cut.cat.categories
# new_df = pd.DataFrame(
#     df["NPS"].groupby(cut, observed=True).sum().reindex(cut_index, fill_value=0)
# )
np_cps, *_ = np.histogram(L_array, bins=bins)

# For SI Fig 7, adjust L_array so total sim counts approx. = avg. experimental counts
cps_sum = np_cps.sum()
exp_sum = 1038527.6
cps_corr_factor = exp_sum / cps_sum
print(cps_corr_factor)
np_cps = np_cps * cps_corr_factor
print(np_cps.sum())

# old_index = new_df.index
# if not isinstance(old_index, pd.IntervalIndex):
#     raise RuntimeError(
#         f"DataFrame created from cut/groupby for {test_file.name} "
#         + "was not an IntervalIndex as expected"
#     )
# mids = old_index.mid.to_series(index=cut_index)

# np_cps = new_df["NPS"].to_numpy().reshape(-1, 1)
# np_Ls = mids.to_numpy()
np_cps = np_cps.reshape(-1, 1)
np_Ls = (bins[1:] + bins[:-1]) / 2

ddN = NDHistogram(np_cps, [np_Ls, np.ones(1)])

In [ ]:
# print(R.counts.shape)
# sigma = NDHistogram(np.sqrt(N.counts), N.midpoints)
# new_R, new_N, new_phi, new_sigma = clean_data(R, N, NDHistogram(
#     np.ones((1, R.shape[1])),
#     [np.ones(1), R.midpoints[1]]
# ), sigma)
# print(new_R.shape)
# print(new_N.shape)
# print(new_phi.shape)
# print(new_sigma.shape)

In [ ]:
mono_phi, _ = unfold_spectrum(
    R,
    monoN,
    L_cut=0.05,
    # tolerance=0.0000001,
    max_iterations=1000,
)

In [ ]:
dd_phi, _ = unfold_spectrum(
    R,
    ddN,
    L_cut=0.05,
    # tolerance=0.0000001,
    max_iterations=1000,
)

In [ ]:
mono_phi_flat = mono_phi.counts.reshape(-1)
mono_phi_mids = mono_phi.midpoints[1]

dd_phi_flat = dd_phi.counts.reshape(-1)
dd_phi_mids = dd_phi.midpoints[1]

In [ ]:
base_xerr = [(1.798, (0.31000000000000005, 0.6200000000000001)),
 (1.86, (0.31000000000000005, 0.558)),
 (1.922, (0.3719999999999999, 0.558)),
 (1.984, (0.3719999999999999, 0.496)),
 (2.046, (0.3719999999999999, 0.496)),
 (2.108, (0.3720000000000001, 0.496)),
 (2.17, (0.43399999999999994, 0.43400000000000016)),
 (2.232, (0.3720000000000001, 0.3719999999999999)),
 (2.294, (0.3720000000000001, 0.3719999999999999)),
 (2.418, (0.43400000000000016, 0.31000000000000005)),
 (2.48, (0.496, 0.31000000000000005)),
 (2.604, (0.3719999999999999, 0.4339999999999997)),
 (2.666, (0.4339999999999997, 0.3719999999999999)),
 (2.79, (0.496, 0.3719999999999999)),
 (2.852, (0.3719999999999999, 0.43400000000000016)),
 (2.914, (0.43400000000000016, 0.3719999999999999)),
 (2.976, (0.496, 0.18599999999999994)),
 (3.038, (0.496, 0.1860000000000004)),
 (3.1, (0.5580000000000003, 0.18599999999999994)),
 (3.162, (0.496, 0.18599999999999994)),
 (3.224, (0.5580000000000003, 0.18599999999999994)),
 (3.286, (0.5579999999999998, 0.18599999999999994))]

In [ ]:
mono_xerr = []
for mid in mono_phi_mids:
    isclose = [np.isclose(x, mid) for x, _ in base_xerr]
    true_idx = [i for i, x in enumerate(isclose) if x]
    if len(true_idx) == 0:
        mono_xerr.append((np.nan, np.nan))
        continue
    idx = true_idx[0]
    _, errorbar = base_xerr[idx]
    mono_xerr.append(errorbar)
mono_xerr = list(zip(*mono_xerr))

dd_xerr = []
for mid in mono_phi_mids:
    isclose = [np.isclose(x, mid) for x, _ in base_xerr]
    true_idx = [i for i, x in enumerate(isclose) if x]
    if len(true_idx) == 0:
        dd_xerr.append((np.nan, np.nan))
        continue
    idx = true_idx[0]
    _, errorbar = base_xerr[idx]
    dd_xerr.append(errorbar)
dd_xerr = list(zip(*dd_xerr))

In [ ]:
mono_sigma_counts = np.sqrt(monoN.counts)
monoN_yerr_plus = NDHistogram(monoN.counts + 3 * mono_sigma_counts, monoN.midpoints)
monoN_yerr_minus = NDHistogram(monoN.counts - 3 * mono_sigma_counts, monoN.midpoints)

monophi_yerr_plus, _ = unfold_spectrum(
    R,
    monoN_yerr_plus,
    L_cut=0.05,
    # tolerance=0.0000001,
    max_iterations=1000,
)
monophi_yerr_minus, _ = unfold_spectrum(
    R,
    monoN_yerr_minus,
    L_cut=0.05,
    # tolerance=0.0000001,
    max_iterations=1000,
)
mono_yerr = [
    mono_phi_flat - monophi_yerr_minus.counts.reshape(-1),
    monophi_yerr_plus.counts.reshape(-1) - mono_phi_flat
]

In [ ]:
dd_sigma_counts = np.sqrt(ddN.counts)
dd_sigma = NDHistogram(dd_sigma_counts, ddN.midpoints)
ddN_yerr_plus = NDHistogram(ddN.counts + 3 * dd_sigma_counts, ddN.midpoints)
ddN_yerr_minus = NDHistogram(ddN.counts - 3 * dd_sigma_counts, ddN.midpoints)

ddphi_yerr_plus, _ = unfold_spectrum(
    R,
    ddN_yerr_plus,
    L_cut=0.05,
    # tolerance=0.0000001,
    max_iterations=1000,
    sigma=dd_sigma
)
ddphi_yerr_minus, _ = unfold_spectrum(
    R,
    ddN_yerr_minus,
    L_cut=0.05,
    # tolerance=0.0000001,
    max_iterations=1000,
    sigma=dd_sigma
)
ddphi_yerr_plus_flat = ddphi_yerr_plus.counts.reshape(-1)
ddphi_yerr_minus_flat = ddphi_yerr_minus.counts.reshape(-1)

plus_is_larger = ddphi_yerr_plus_flat >= ddphi_yerr_minus_flat
plus_gt_phi = (ddphi_yerr_plus_flat >= dd_phi_flat) | (np.isnan(dd_phi_flat))
minus_lt_phi = (ddphi_yerr_minus_flat <= dd_phi_flat) | (np.isnan(dd_phi_flat))
ddphi_yerr_larger = ddphi_yerr_plus_flat.copy()
ddphi_yerr_larger[~plus_gt_phi] = ddphi_yerr_minus_flat[~plus_gt_phi]
ddphi_yerr_smaller = ddphi_yerr_minus_flat.copy()
ddphi_yerr_smaller[~minus_lt_phi] = ddphi_yerr_plus_flat[~minus_lt_phi]

dd_yerr_up = np.nan_to_num(ddphi_yerr_larger - dd_phi_flat, nan=np.nan)
dd_yerr_down = np.nan_to_num(dd_phi_flat - ddphi_yerr_smaller, nan=np.nan)
dd_yerr_up[dd_yerr_up < 0] = 0
dd_yerr_down[dd_yerr_down < 0] = 0

dd_xerr_left, _ = dd_xerr
dd_yerr_down[np.isnan(dd_xerr_left)] = np.nan
dd_yerr_up[np.isnan(dd_xerr_left)] = np.nan

dd_yerr = [dd_yerr_down, dd_yerr_up]

In [ ]:
#linalg normalization test
# mono_norm = np.linalg.norm(mono_phi_flat[~np.isnan(mono_phi_flat)])
# mono_phi_normed = mono_phi_flat/mono_norm

dd_norm = np.linalg.norm(dd_phi_flat[~np.isnan(dd_phi_flat)])
dd_phi_normed = dd_phi_flat / dd_norm
dd_N_counts = np.nansum(ddN.counts)
dd_phi_norm_sum = np.nansum(dd_phi_normed)
count_adj_factor = dd_N_counts / dd_phi_norm_sum
dd_phi_normed = dd_phi_normed * count_adj_factor

d_detector = 12.7  # cm
cs_area = pi * d_detector * d_detector / 4
dd_phi_fluence = dd_phi_normed / cs_area

dd_yerr = [
    ((err / dd_norm) * count_adj_factor) / cs_area
    for err in dd_yerr
]

In [ ]:
figsize=(9,6)
fontsize = 20
fig, ax = plt.subplots(figsize=figsize, dpi=300)
# ax.errorbar(
#     mono_phi_mids, mono_phi_normed,
#     xerr=mono_errorbars,
#     marker="o", markersize=3,
#     label="2.45 MeV monoenergetic simulation"
#     # label="E1"
#     # label="E3"
# )
ax.errorbar(
    dd_phi_mids,
    # dd_phi_normed,
    dd_phi_fluence,
    xerr=dd_xerr,
    yerr=dd_yerr,
    marker="o", markersize=3, ecolor="black",
    markerfacecolor="red",
    markeredgecolor="red",
    label="DD fusion simulation"
    # label="E2"
    # label="E4"
)
# ax.plot(phi.midpoints[peaks], phi.counts[peaks], marker="x", markersize=8)
# if len(peaks) > 0:
#     ax.vlines(x=phi_mids[peaks], ymin=0, ymax=phi_flat[peaks], colors="red", linestyles="dotted")
#     for peak_x, peak_y in zip(phi_mids[peaks], phi_flat[peaks]):
#         ax.annotate(f"{peak_x} MeV", (peak_x, peak_y), (5, 0), textcoords="offset fontsize", arrowprops={"width": 2}, verticalalignment="center")
ax.set(
    # ylim=(0, 0.05),
    # title="Unfolded Spectrum (Simulated AmBe Neutrons)"
)
ax.set_xlabel("Neutron energy (MeV)", fontsize=fontsize)
ax.set_ylabel(r"Neutron fluence (1/cm${^2}$)", fontsize=fontsize)
# ax.set_title("Using Numpy")
# ax.legend()
# ax.set_yscale("log")
ax.tick_params(labelsize=fontsize)
plt.show()

In [ ]:
input("Processing done, hit Enter to finish")
helpers.stop()

In [ ]:
# TODO finish brute force method
def safe_divide(a, b):
    try:
        with np.errstate(divide="ignore", invalid="ignore"):
            return np.nan_to_num(a / b, nan=np.nan, posinf=np.nan, neginf=np.nan)
    except ZeroDivisionError:
        return np.nan


def r_phi_sum(r, phi):
    _m, _n = r.shape
    result = np.zeros((_m, 1))

    for i in range(_m):
        j_sum = 0
        for j in range(_n):
            product = r[i][j] * phi[0][j]
            if not np.isnan(product):
                j_sum += product
        result[i][0] = j_sum

    return result


def weight_looped(r, n, phi, sigma):
    _m, _n = r.shape
    result = np.zeros((_m, _n))
    _r_phi_sum = r_phi_sum(r, phi)

    for i in range(_m):
        for j in range(_n):
            l_numer = r[i][j] * phi[0][j]
            l_denom = _r_phi_sum[i][0]
            r_numer = np.square(n[i][0])
            r_denom = np.square(sigma[i][0])
            l_frac = safe_divide(l_numer, l_denom)
            r_frac = safe_divide(r_numer, r_denom)
            result[i][j] = l_frac * r_frac
    return result


def next_phi_looped(r, n, phi, sigma):
    _m, _n = r.shape
    result = np.zeros((1, _n))
    w = weight_looped(r, n, phi, sigma)
    _r_phi_sum = r_phi_sum(r, phi)

    for j in range(_n):
        numer_i_sum = 0
        denom_i_sum = 0
        for i in range(_m):
            _w = w[i][j]
            ln_frac = safe_divide(n[i][0], _r_phi_sum[i][0])
            with np.errstate(divide="ignore", invalid="ignore"):
                ln_result = np.nan_to_num(np.log(ln_frac), nan=np.nan, posinf=np.nan, neginf=np.nan)
            numer_product = _w * ln_result
            if not np.isnan(numer_product):
                numer_i_sum += numer_product
            if not np.isnan(_w):
                denom_i_sum += _w
        exp_frac = safe_divide(numer_i_sum, denom_i_sum)
        with np.errstate(divide="ignore", invalid="ignore"):
            exp_result = np.nan_to_num(np.exp(exp_frac), nan=np.nan, posinf=np.nan, neginf=np.nan)
        result[0][j] = phi[0][j] * exp_result

    return result


def stopping_criteria_looped(r, n, phi, sigma):
    _m, _n = r.shape
    DOF = (_m-1)*(_n-1)
    _r_phi_sum = r_phi_sum(r, phi)

    i_sum = 0
    for i in range(_m):
        delta = _r_phi_sum[i][0] - n[i][0]
        d_sq = np.square(delta)
        frac = safe_divide(d_sq, np.square(sigma[i][0]))
        if not np.isnan(frac):
            i_sum += frac
    return i_sum / DOF


def unfold_spectrum_looped(r, n, phi0=None, sigma=None, L_cut=None, tolerance=0.01, max_iters=500):
    if phi0 is None:
        phi0 = NDHistogram(np.ones((1, r.shape[1])), [np.ones(1), r.midpoints[1]])
    if sigma is None:
        sigma = NDHistogram(np.sqrt(n.counts), n.midpoints)

    _r, _n, _sigma = cut_low_l(r, n, sigma=sigma, L_cut=L_cut)

    r_counts = _r.counts.copy()
    n_counts = _n.counts.copy()
    sigma_counts = _sigma.counts.copy()
    phi_counts = phi0.counts.copy()

    iters = 0
    chis = []
    phis = []
    weights = []
    errors = []
    iter_text_len = len(str(max_iters))
    
    chi_n = stopping_criteria_looped(r_counts, n_counts, phi_counts, sigma_counts)
    chi_last = chi_n
    delta_chi_last = 1
    delta_delta = 1
    
    while delta_delta > tolerance:
        _w = weight_looped(r_counts, n_counts, phi_counts, sigma_counts)
        phi_counts = next_phi_looped(r_counts, n_counts, phi_counts, sigma_counts)
        chi_n = stopping_criteria_looped(r_counts, n_counts, phi_counts, sigma_counts)
    
        delta_chi = chi_n - chi_last
        delta_delta = abs(delta_chi - delta_chi_last)
        chi_last = chi_n
        delta_chi_last = delta_chi
    
        chis.append(chi_n)
        phis.append(phi_counts)
        weights.append(_w)
        errors.append(delta_delta)
    
        if iters % 10 == 0:
            print(
                f"Iter. {iters: {iter_text_len}d}: chi = {chi_n:.3g}, rel_rate = {delta_delta: .3g}"
            )
        iters += 1
        if iters >= max_iters:
            break

    unfolding_info = UnfoldingProcessInfo(errors=errors, chis=chis, phis=phis, weights=weights)
    return phi_counts, unfolding_info

In [ ]:
testr = np.array([list(range(i, i+5)) for i in range(1,7)])
testphi = np.array([[1, 2, 3, 4, 5]])
_r_phi_sum = r_phi_sum(testr, testphi)
print(_r_phi_sum)

In [ ]:
testn = np.array([[i] for i in range(3, 21, 3)])
testsigma = np.sqrt(testn)

_m, _n = testr.shape
result = np.zeros((_m, _n))

for i in range(_m):
    for j in range(_n):
        prod = testr[i][j] * testphi[0][j]
        l_frac = prod / _r_phi_sum[i][0]
        # result[i][j] = frac
        numer = np.square(testn[i][0])
        denom = np.square(testsigma[i][0])
        r_frac = numer / denom
        frac_prod = l_frac * r_frac
        result[i][j] = frac_prod

result

In [ ]:
weight_looped(testr, testn, testphi, testsigma)

In [ ]:
# phi0_counts = np.ones((1, R.shape[1]))

# mono_sigma = NDHistogram(np.sqrt(monoN.counts), monoN.midpoints)
# _R, _monoN, _mono_sigma = cut_low_l(R, monoN, L_cut=0.05, sigma=mono_sigma)

# _r = _R.counts.copy()
# _monon = _monoN.counts.copy()
# _monosigma = _mono_sigma.counts.copy()

# _SC = stopping_criteria_looped(_r, _monon, phi0_counts, _monosigma)
# print(_SC)
# _W = weight_looped(_r, _monon, phi0_counts, _monosigma)
# _phi = next_phi_looped(_r, _monon, phi0_counts, _monosigma)
# print(_phi)
# _SC = stopping_criteria_looped(_r, _monon, _phi, _monosigma)
# print(_SC)

In [ ]:
# sc_tol = 0.01
# dd_tol = 0.01
# max_iters = 500

# phi0_counts = np.ones((1, R.shape[1]))
# _phi = phi0_counts.copy()
# mono_sigma = NDHistogram(np.sqrt(monoN.counts), monoN.midpoints)
# _R, _monoN, _mono_sigma = cut_low_l(R, monoN, L_cut=0.05, sigma=mono_sigma)

# _r = _R.counts.copy()
# _monon = _monoN.counts.copy()
# _monosigma = _mono_sigma.counts.copy()

# iters = 0
# chis = []
# phis = []
# weights = []
# errors = []
# iter_text_len = len(str(max_iters))

# chi_n = stopping_criteria_looped(_r, _monon, _phi, _monosigma)
# chi_last = chi_n
# delta_chi_last = 1
# delta_delta = 1

# while delta_delta > dd_tol:
#     _w = weight_looped(_r, _monon, _phi, _monosigma)
#     _phi = next_phi_looped(_r, _monon, _phi, _monosigma)
#     chi_n = stopping_criteria_looped(_r, _monon, _phi, _monosigma)

#     delta_chi = chi_n - chi_last
#     delta_delta = abs(delta_chi - delta_chi_last)
#     chi_last = chi_n
#     delta_chi_last = delta_chi

#     chis.append(chi_n)
#     phis.append(_phi)
#     weights.append(_w)
#     errors.append(delta_delta)

#     if iters % 10 == 0:
#         print(
#             f"Iter. {iters: {iter_text_len}d}: chi = {chi_n:.3g}, rel_rate = {delta_delta: .3g}"
#         )
#     iters += 1
#     if iters >= max_iters:
#         break

In [ ]:
brute_mono_phi, mono_unfolding_info = unfold_spectrum_looped(R, monoN, L_cut=0.05)
brute_dd_phi, dd_unfolding_info = unfold_spectrum_looped(R, ddN, L_cut=0.05)

In [ ]:
len(mono_unfolding_info["phis"])

In [ ]:
brute_mono_phi_flat = brute_mono_phi.reshape(-1)
brute_dd_phi_flat = brute_dd_phi.reshape(-1)
brute_phi_mids = R.midpoints[1]

In [ ]:
figsize=(9,6)
dd_scaling = 1
fontsize = 20
fig, ax = plt.subplots(figsize=figsize, dpi=300)
ax.plot(brute_phi_mids, brute_mono_phi_flat, marker="o", markersize=3, label="2.45 MeV monoenergetic simulation")
ax.plot(brute_phi_mids, brute_dd_phi_flat * dd_scaling, marker="o", markersize=3, label="DD fusion simulation")
# ax.plot(phi.midpoints[peaks], phi.counts[peaks], marker="x", markersize=8)
# if len(peaks) > 0:
#     ax.vlines(x=phi_mids[peaks], ymin=0, ymax=phi_flat[peaks], colors="red", linestyles="dotted")
#     for peak_x, peak_y in zip(phi_mids[peaks], phi_flat[peaks]):
#         ax.annotate(f"{peak_x} MeV", (peak_x, peak_y), (5, 0), textcoords="offset fontsize", arrowprops={"width": 2}, verticalalignment="center")
ax.set(
    # ylim=(0, 0.05),
    # title="Unfolded Spectrum (Simulated AmBe Neutrons)"
)
ax.set_xlabel("E (MeV)", fontsize=fontsize)
ax.set_ylabel("Normalized counts", fontsize=fontsize)
ax.set_title("Using Python loop")
# ax.set_yscale("log")
ax.tick_params(labelsize=fontsize)
plt.show()